In [4]:
from helper import *
import json

def generate_right_json(G, collaborators, directly_influenced, indirectly_influenced, output_path):
    """
    Create a JSON file for the right side (collaborators and influenced artists)
    with the same structure as the left-side JSON.

    collaborators: dict {artist_node: set of (work, date_str, role)}
    directly_influenced: dict {artist_node: set of (etype, ctype, source_work, inf_work, date_str, score, node_type)}
    indirectly_influenced: same as directly_influenced
    """
    nodes = []
    links = []
    node_ids = set()

    # Helper to add node
    def add_node(node_id, node_name, node_type, first_year):
        if node_id not in node_ids:
            nodes.append({
                "id": node_id,
                "type": node_type,
                "name": node_name,
                "firstYear": first_year
            })
            node_ids.add(node_id)

    # --- 1. Collaborators ---
    for artist, events in collaborators.items():
        name = G.nodes[artist].get('name')
        # Find earliest year
        years = []
        for work, date_str, role, genre in events:
            if date_str:
                years.append(int(date_str))
        if not years:
            continue
        first_year = min(years)
        add_node(artist, name, "collaborator", first_year)

        # Add a link for each collaboration event
        for work, date_str, role, genre in events:
            year = int(date_str)
            links.append({
                "source": "sailor",
                "target": artist,
                "year": year,
                "type": "collaboration",
                "role": role,
                "work": work,
                "score": 1          # each collaboration counts as 1 point
            })

    # --- 2. Direct influences ---
    for artist, events in directly_influenced.items():
        name = G.nodes[artist].get('name')
        years = []
        for ev in events:
            # ev: (etype, ctype, source_work, inf_work, date_str, score, node_type, genre)
            if ev[4]:
                years.append(int(ev[4]))
        if not years:
            continue
        first_year = min(years)
        add_node(artist, name, "influenced", first_year)

        for ev in events:
            etype, ctype, source_work, inf_work, date_str, score, node_type, genre = ev
            year = int(date_str)
            links.append({
                "source": "sailor",
                "target": artist,
                "year": year,
                "type": "influence",
                "score": score,
                "work": inf_work,
                "sailor_work": source_work
            })

    # --- 3. Indirect influences ---
    for artist, events in indirectly_influenced.items():
        # If already added as a node (e.g., also direct), just add links
        name = G.nodes[artist].get('name')
        if artist not in node_ids:
            years = []
            for ev in events:
                if ev[4]:
                    years.append(int(ev[4]))
            if not years:
                continue
            first_year = min(years)
            add_node(artist, name, "influenced", first_year)

        for ev in events:
            etype, ctype, source_work, inf_work, date_str, score, node_type = ev
            year = int(date_str)
            links.append({
                "source": "sailor",
                "target": artist,
                "year": year,
                "type": "influence",
                "score": score,
                "work": inf_work,
                "sailor_work": source_work
            })

    # Write to JSON
    with open(output_path + "data_task1-2.json", "w") as f:
        json.dump({"nodes": nodes, "links": links}, f, indent=2)

def find_indirectly_influenced(G, sailor_node, directly_influenced_dict):
    """
    Finds artists influenced indirectly via chains of influence edges.
    directly_influenced_dict: output from find_directly_influenced (keys are directly influenced artists)
    Returns a dict in the same format: {artist_node: {(etype, ctype, source_work, inf_work, date, score)}}
    """
    artists_influenced_indirectly = {}
    visited = set(directly_influenced_dict.keys()) | {sailor_node}
    # print(visited)
    # queue holds (artist, hop) – hop is the current level (starting at 2 for first indirect step)
    queue = list(directly_influenced_dict.keys())
    # print(queue)

    influence_types = ['InStyleOf', 'InterpolatesFrom', 'CoverOf', 'LyricalReferenceTo', 'DirectlySamples']
    person_roles = ['PerformerOf', 'ComposerOf', 'ProducerOf', 'LyricistOf']
    sum_of_influenced_score = 0 ## how much artist was influenced directly by Sailor e.g (1 + 0.5)=1.5
    while queue:
        artist = queue.pop(0)
        print(f"artist:{G.nodes[artist].get('name')}")
        # Get all works by this artist (including band works)
        artist_works, bands = find_all_artist_works(G, artist)
        # print(f"artist_works:{artist_works}\n\n")
        artist_roles = find_role_of_person(G, artist)
        if bands:
            artist_roles.add('PerformerOf')
        # print(f"artist_roles:{artist_roles}\n\n")
        for item in directly_influenced_dict[artist]:
            sum_of_influenced_score += item[5]
            
        for work in artist_works:
            work_date = G.nodes[work].get('release_date')
            for etype in influence_types:
                # incoming edges to 'work' = works influenced by it
                for inf_work in get_neighbors_by_edge_type(G, work, etype, 'in'):
                    inf_work_date = G.nodes[inf_work].get('release_date')
                    if inf_work_date and int(inf_work_date) >= int(work_date):
                        for ctype in person_roles:
                            for creator in get_neighbors_by_edge_type(G, inf_work, ctype, 'in'):
                                if creator == sailor_node or creator in visited:
                                    continue
                                node_type = G.nodes[creator].get('Node Type')
                                # Compute role factor based on the influencer's (artist) roles
                                role_factor = 1/2 if ctype in artist_roles else 1/3
                                score = role_factor * sum_of_influenced_score 
                                if node_type == 'Person':
                                        if etype == 'InterpolatesFrom':
                                            if ctype == 'ComposerOf':
                                                    add_influence(creator, etype, ctype, work, inf_work, inf_work_date, score, node_type, artists_influenced_indirectly)    
                                        else: #if etype=InStyleOf, CoverOf,....
                                            # print("yes")
                                            add_influence(creator, etype, ctype, work, inf_work,inf_work_date, score, node_type, artists_influenced_indirectly)
                                
                                elif node_type == 'MusicalGroup':
                                        for member in get_neighbors_by_edge_type(G, creator, member_type, 'in'):
                                            if member != sailor_node:
                                                add_influence(member, etype, ctype, work, inf_work, inf_work_date, score, node_type, artists_influenced_indirectly)
                
                
    return artists_influenced_indirectly



def main():
    G = read_data_from_json_to_graph(data_file_path)
    n, a = find_sailor_shift(G)
    print (n)
    collaborators = find_collaborators_of_artist(G, n)
    # print(f"collaborators:{collaborators}")
    # print (len(collaborators))
    # print(f"# of collaborators:{len(collaborators)}")
    artists_influenced_directly = find_directly_influenced(G, n)
    # print(f"influencesss:{artists_influenced_directly}")
    artists_influenced_indirectly = find_indirectly_influenced(G, n, artists_influenced_directly)
    generate_right_json(G, collaborators, artists_influenced_directly , artists_influenced_indirectly, 'data/' )
    print(f"indirect:{artists_influenced_indirectly}")
    indirect_influenced_rows = []
    for artist, tuple_set in artists_influenced_directly.items():
        artist_name = G.nodes[artist].get('name')
        for tuple in tuple_set:
            work = tuple[3]
            # print(work)
            genre = G.nodes[work].get('genre')
            indirect_influenced_rows.append({
                'name' : artist_name,
                'work' : tuple[3],
                'sailor_work' : tuple[2],
                'genre' : genre,
                'date' : tuple[4],
                'score' : tuple[5]
            })
    df_indirect_influenced = pd.DataFrame(indirect_influenced_rows)
    print(df_indirect_influenced)
    # collaborator_rows = []
    # for collaborator, tuple_set in collaborators.items():
    #     collaborator_name = G.nodes[collaborator].get('name')
    #     for tuple in tuple_set:
    #         collaborator_rows.append({
    #             'name' : collaborator_name,
    #             'work' : tuple[0],
    #             'date' : tuple[1],
    #             'role' : tuple[2],
    #             'influence': 'direct'
    #     })
    # df_collaborators = pd.DataFrame(collaborator_rows)
    # # print(df_collaborators)
    
    # # genres = set()  # Use a set to avoid duplicates

    # # for node, data in G.nodes(data=True):
    # #     node_type = data.get('Node Type')
    # #     if node_type in ('Album', 'Song'):
    # #         genre = data.get('genre')
    # #         if genre:
    # #             # If genre could be a list (multiple genres), handle accordingly
    # #             if isinstance(genre, list):
    # #                 genres.update(genre)
    # #             else:
    # #                 genres.add(genre)

    # # print(genres)
    # sailor_works, bands = find_all_artist_works(G, n)
    # # print(len(bands))
    # # print(f"# of Sailor works:{len(sailor_works)}")
    # # for sailor_work in sailor_works:
    #     # print(f"{G.nodes[sailor_work].get('genre')} in year of {G.nodes[sailor_work].get('release_date')} and notable={G.nodes[sailor_work].get('notable')}\n")
        
    # genres = {}
    # for node, data in G.nodes(data=True):
    #     if data.get('Node Type') in ('Song', 'Album'):
    #         genre = data.get('genre')
    #         if genre:
    #             genres[genre] = genres.get(genre, 0) + 1
    # # print(genres)
    # # print(len(genres))
if __name__ == "__main__":
    main()    
                    
            
        
    

Type of data: <class 'dict'>
17255
artist:Chloe Montgomery
artist:Nathaniel Brooks
artist:Clara Davis
artist:Scarlett Moon
artist:N.V. Blake
artist:Ivy Steele
artist:Nate Wild
indirect:{}
               name   work  sailor_work                  genre  date  score
0  Chloe Montgomery  17112        17268  Post-Apocalyptic Folk  2031    1.0
1  Nathaniel Brooks  17112        17268  Post-Apocalyptic Folk  2031    1.0
2       Clara Davis  17112        17268  Post-Apocalyptic Folk  2031    1.0
3     Scarlett Moon  17112        17268  Post-Apocalyptic Folk  2031    0.5
4        N.V. Blake  17112        17268  Post-Apocalyptic Folk  2031    0.5
5        Ivy Steele  17112        17268  Post-Apocalyptic Folk  2031    0.5
6         Nate Wild  17112        17268  Post-Apocalyptic Folk  2031    0.5
